# Heart Disease Prediction — Exploratory Data Analysis

This notebook performs a comprehensive exploratory data analysis (EDA) on the heart disease dataset. We explore data distributions, relationships, outliers, and correlations to understand the data before building machine learning models.

**Target Variable:** `target` (0 = No Heart Disease, 1 = Heart Disease)

In [ ]:
# Import libraries
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Paths (notebook is in notebooks/, project root is one level up)
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("EDA.ipynb")))
if not os.path.exists(os.path.join(BASE_DIR, "dataset", "heart.csv")):
    BASE_DIR = os.path.dirname(os.getcwd())

DATASET_PATH = os.path.join(BASE_DIR, "dataset", "heart.csv")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

TARGET = "target"

## Dataset Overview

Load the dataset and inspect its structure, dimensions, and basic statistics.

In [ ]:
# Load dataset
df = pd.read_csv(DATASET_PATH)

print("First 5 rows:")
display(df.head())

print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nDescriptive Statistics:")
display(df.describe())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

## Missing Values Analysis

Check for null/missing values across all columns.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "Column": missing.index,
    "Missing Count": missing.values,
    "Missing %": missing_pct.values,
})
display(missing_df)

if missing.sum() == 0:
    print("✅ No missing values found in the dataset.")
else:
    plt.figure(figsize=(10, 4))
    sns.barplot(x="Column", y="Missing %", data=missing_df[missing_df["Missing Count"] > 0])
    plt.title("Missing Values by Column")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Target Distribution

Analyze the balance of heart disease vs. no heart disease classes.

In [ ]:
target_counts = df[TARGET].value_counts()
print("Target class distribution:")
print(target_counts)
print(f"\nProportions:\n{df[TARGET].value_counts(normalize=True)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ["#2ECC71", "#E74C3C"]
labels = ["No Heart Disease (0)", "Heart Disease (1)"]

axes[0].bar(labels, target_counts.values, color=colors, edgecolor="white")
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")
axes[0].set_title("Target Distribution (Count)", fontweight="bold")
axes[0].set_ylabel("Count")

axes[1].pie(
    target_counts.values, labels=labels, autopct="%1.1f%%",
    colors=colors, startangle=90, explode=(0.02, 0.02),
)
axes[1].set_title("Target Distribution (%)", fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "target_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

## Histograms

Distribution of each feature in the dataset.

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET]
n_cols = 3
n_rows = (len(feature_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="#3498DB", edgecolor="white")
    axes[i].set_title(col, fontweight="bold")

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Histograms", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "feature_histograms.png"), dpi=150, bbox_inches="tight")
plt.show()

## Boxplots

Boxplots help identify outliers and compare feature distributions across target classes.

In [ ]:
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features = [c for c in numeric_features if c != TARGET]

fig, axes = plt.subplots(4, 3, figsize=(14, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.boxplot(x=TARGET, y=col, data=df, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} by Target", fontweight="bold")
    axes[i].set_xlabel("Target")

for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Boxplots by Target Class", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Correlation Heatmap

Examine correlations between features to identify redundant or strongly related variables.

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=0.5, square=True,
)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "correlation_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## Pairplots

Pairwise relationships between key numerical features colored by target class.

In [ ]:
pair_features = ["age", "trestbps", "chol", "thalach", "oldpeak", TARGET]
pair_df = df[pair_features].copy()
pair_df[TARGET] = pair_df[TARGET].map({0: "No Disease", 1: "Heart Disease"})

g = sns.pairplot(
    pair_df, hue=TARGET, diag_kind="kde",
    palette={"No Disease": "#2ECC71", "Heart Disease": "#E74C3C"},
    plot_kws={"alpha": 0.6, "s": 30},
    height=2.2,
)
g.fig.suptitle("Pairplot of Key Features", y=1.02, fontsize=14, fontweight="bold")
plt.show()

## Feature Distributions

Violin plots showing feature distributions split by target class.

In [ ]:
key_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

fig, axes = plt.subplots(1, len(key_features), figsize=(16, 4))
for i, col in enumerate(key_features):
    sns.violinplot(x=TARGET, y=col, data=df, ax=axes[i], palette="Set2", inner="quartile")
    axes[i].set_title(col, fontweight="bold")
    axes[i].set_xlabel("Target")

fig.suptitle("Feature Distributions by Target", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

## Outlier Detection

Use the IQR method to detect potential outliers in numerical features.

In [ ]:
outlier_summary = []

for col in numeric_features:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary.append({
        "Feature": col,
        "Outlier Count": len(outliers),
        "Outlier %": round(len(outliers) / len(df) * 100, 2),
        "Lower Bound": round(lower, 2),
        "Upper Bound": round(upper, 2),
    })

outlier_df = pd.DataFrame(outlier_summary)
display(outlier_df)

plt.figure(figsize=(10, 5))
sns.barplot(x="Feature", y="Outlier Count", data=outlier_df, color="#E74C3C", edgecolor="white")
plt.title("Outlier Count by Feature (IQR Method)", fontweight="bold")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Feature Relationships with Target

Analyze how each feature relates to the heart disease target variable.

In [ ]:
# Correlation with target
target_corr = df.corr()[TARGET].drop(TARGET).sort_values(ascending=False)
print("Feature correlation with target:")
print(target_corr)

plt.figure(figsize=(10, 6))
colors = ["#E74C3C" if v > 0 else "#3498DB" for v in target_corr.values]
plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor="white")
plt.xlabel("Correlation with Target")
plt.title("Feature Correlation with Heart Disease", fontweight="bold")
plt.axvline(x=0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

# Categorical feature analysis
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    ct = pd.crosstab(df[col], df[TARGET], normalize="index") * 100
    ct.plot(kind="bar", stacked=False, ax=axes[i], color=["#2ECC71", "#E74C3C"], edgecolor="white")
    axes[i].set_title(f"{col} vs Target", fontweight="bold")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Percentage")
    axes[i].legend(["No Disease", "Heart Disease"])
    axes[i].tick_params(axis="x", rotation=0)

fig.suptitle("Categorical Features vs Target", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Summary

Key findings from this EDA:

- Dataset contains patient medical records with 13 features and a binary target
- Target class distribution should be checked for balance
- Correlation heatmap reveals relationships between clinical features
- Boxplots and IQR analysis help identify outliers
- Features like `cp`, `thalach`, `exang`, and `oldpeak` often show strong relationships with heart disease

**Next step:** Run `python train_model.py` to train and evaluate machine learning models.